# 🎙 Озвучка Nuvora вашим голосом

1. Сверху: **Среда выполнения → Сменить среду выполнения → T4 GPU → Сохранить**.
2. В ячейке ниже вставьте **код загрузки** — его покажет на сервере команда `bash /opt/nuvora/deploy/nuva-upload-code.sh`.
3. **Среда выполнения → Выполнить все.** Обучение идёт около часа. Не закрывайте вкладку.
4. Когда внизу появится «✅ Голос загружен», нажмите «Прослушать» у любого поста — это будет ваш голос.

Если обучение идёт слишком долго — нажмите ■ (остановить) у ячейки обучения и запустите ячейки ниже: возьмётся последнее сохранение.

In [ ]:
SERVER = "https://nvurl.ru"   # адрес вашего сайта
CODE = ""                     # ← вставьте сюда код загрузки в кавычках
EPOCHS = 1500                 # сколько кругов обучения: больше — похожее, но дольше (1500 ≈ час на T4)
BASE_VOICE = "denis"          # от какого готового голоса доучивать: denis или dmitri (мужские), irina (женский)
assert CODE, "Вставьте код загрузки в CODE (команда на сервере: bash /opt/nuvora/deploy/nuva-upload-code.sh)"

## 1. Устанавливаю программы (~5 минут)

In [ ]:
!apt-get -qq install -y build-essential cmake ninja-build ffmpeg > /dev/null
!git clone -q --depth 1 --branch v1.8.0 https://github.com/OHF-Voice/piper1-gpl.git || git clone -q --depth 1 https://github.com/OHF-Voice/piper1-gpl.git
%cd piper1-gpl
!pip install -q -e '.[train]'
!./build_monotonic_align.sh
!python3 setup.py build_ext --inplace > /dev/null
%cd ..
import torch
assert torch.cuda.is_available(), "Нет видеокарты: Среда выполнения → Сменить среду выполнения → T4 GPU"
print("Видеокарта:", torch.cuda.get_device_name(0))

## 2. Скачиваю ваши записи с сайта

In [ ]:
import io, os, subprocess, zipfile, requests
r = requests.get(SERVER + "/api/voice/dataset", headers={"x-upload-code": CODE}, timeout=300)
assert r.status_code == 200, f"Сервер ответил {r.status_code}: {r.text[:200]} — проверьте код загрузки"
os.makedirs("voice/wavs", exist_ok=True)
z = zipfile.ZipFile(io.BytesIO(r.content))
rows = []
for line in z.read("metadata.csv").decode("utf-8").splitlines():
    if "|" not in line:
        continue
    n, text = line.split("|", 1)
    src = next((x for x in z.namelist() if x.startswith(f"wavs/{n}.")), None)
    if not src:
        continue
    raw = "voice/" + os.path.basename(src)
    open(raw, "wb").write(z.read(src))
    out = f"voice/wavs/{n}.wav"
    ok = subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-i", raw, "-ac", "1", "-ar", "22050", "-af", "loudnorm", out]).returncode == 0
    if ok:
        rows.append(f"{n}.wav|{text}")
open("voice/metadata.csv", "w", encoding="utf-8").write("\n".join(rows) + "\n")
print("Записей:", len(rows))
assert len(rows) >= 30, "Слишком мало записей — запишите хотя бы 30 фраз в «Студии голоса»"

## 3. Беру готовый русский голос, от которого будем доучивать

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files
files = [f for f in list_repo_files("rhasspy/piper-checkpoints", repo_type="dataset") if f.startswith("ru/ru_RU/") and "/medium/" in f and f.endswith(".ckpt")]
print("Есть русские контрольные точки:", files)
pick = next((f for f in files if f"/{BASE_VOICE}/" in f), None) or next((f for f in files if "/denis/" in f or "/dmitri/" in f), files[0] if files else None)
assert pick, "Не нашлось русской контрольной точки"
CKPT = hf_hub_download("rhasspy/piper-checkpoints", pick, repo_type="dataset")
print("Доучиваю от:", pick)

## 4. Обучение (~1 час). Можно остановить раньше — возьмётся последнее сохранение

In [ ]:
!python3 -m piper.train fit \
  --data.voice_name "nuvora-custom" \
  --data.csv_path voice/metadata.csv \
  --data.audio_dir voice/wavs \
  --model.sample_rate 22050 \
  --data.espeak_voice "ru" \
  --data.cache_dir voice/cache \
  --data.config_path voice/nuvora-custom.onnx.json \
  --data.batch_size 16 \
  --model.warmstart_ckpt "$CKPT" \
  --trainer.max_epochs $EPOCHS \
  --trainer.check_val_every_n_epoch 50

## 5. Перевожу в формат для сайта и слушаю пример

In [ ]:
import glob
ckpts = sorted(glob.glob("lightning_logs/*/checkpoints/last.ckpt"), key=os.path.getmtime) or sorted(glob.glob("lightning_logs/*/checkpoints/*.ckpt"), key=os.path.getmtime)
assert ckpts, "Нет сохранений обучения — запустите ячейку обучения хотя бы на несколько минут"
!python3 -m piper.train.export_onnx --checkpoint "{ckpts[-1]}" --output-file voice/nuvora-custom.onnx
import wave
from piper import PiperVoice
with wave.open("voice/test.wav", "wb") as w:
    PiperVoice.load("voice/nuvora-custom.onnx", "voice/nuvora-custom.onnx.json").synthesize_wav("Привет! Это Nuvora, и теперь посты читает мой голос.", w)
from IPython.display import Audio, display
display(Audio("voice/test.wav"))

## 6. Отправляю голос на сайт

In [ ]:
cfg = open("voice/nuvora-custom.onnx.json", "rb").read()
r = requests.post(SERVER + "/api/voice-model/upload?kind=json", data=cfg, headers={"x-upload-code": CODE}, timeout=120)
assert r.ok, r.text
data = open("voice/nuvora-custom.onnx", "rb").read()
PART = 40 * 1024 * 1024
parts = [data[i:i + PART] for i in range(0, len(data), PART)]
for i, chunk in enumerate(parts):
    for attempt in range(4):
        try:
            r = requests.post(f"{SERVER}/api/voice-model/upload?kind=onnx&part={i}&total={len(parts)}", data=chunk, headers={"x-upload-code": CODE}, timeout=600)
            break
        except Exception as e:
            print("повтор…", e)
    assert r.ok and r.json().get("ok"), r.text
    print(f"часть {i + 1} из {len(parts)} — ок")
print("✅ Голос загружен! Нажмите «Прослушать» у любого поста на сайте.")